# ST3189: Machine Learning — Section 3: Supervised Learning (Regression)

**Candidate Number:** 

---

**Dataset:** `new-players-data-full.csv` (FIFA Players, 18,773 rows × 74 columns)  
**Task:** Predict a player's **transfer market value** (€ millions) using their performance attributes and physical characteristics.  
**Context:** The unsupervised learning section (Section 1) clustered players into 4 positional archetypes. This section builds on that by treating player value as a continuous target and applying regression models to estimate it.

## 3.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 3.2 Load and Inspect Data

In [ ]:
df = pd.read_csv("new-players-data-full.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
df.head(5)

In [ ]:
# Inspect target variable: value (in millions EUR)
print(df[['value', 'overall_rating', 'potential', 'age', 'height_cm', 'weight_kg',
          'international_reputation', 'skill_moves', 'weak_foot']].describe())

## 3.3 Data Cleaning & Feature Selection

In [ ]:
# Keep only relevant features for predicting player value
# Remove identifiers, URLs, non-numeric contextual fields
features_to_keep = [
    'overall_rating', 'potential', 'age', 'height_cm', 'weight_kg',
    'international_reputation', 'skill_moves', 'weak_foot',
    'best_position', 'value'
]

data = df[features_to_keep].copy()

# Drop rows with missing target (value)
data = data.dropna(subset=['value'])

# Encode best_position categorically
le = LabelEncoder()
data['position_encoded'] = le.fit_transform(data['best_position'].fillna('Unknown'))

# Fill remaining NaNs with median
numeric_cols = data.select_dtypes(include=[np.number]).columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

# Remove outliers in value using IQR
Q1 = data['value'].quantile(0.25)
Q3 = data['value'].quantile(0.75)
IQR = Q3 - Q1
data = data[(data['value'] >= Q1 - 1.5*IQR) & (data['value'] <= Q3 + 1.5*IQR)]

print("Cleaned dataset shape:", data.shape)
print("\nValue stats after cleaning:")
print(data['value'].describe())

## 3.4 Exploratory Data Analysis (EDA)

In [ ]:
# Distribution Plot of key numerical features
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
num_features = ['value', 'overall_rating', 'potential', 'age',
                'height_cm', 'weight_kg', 'international_reputation',
                'skill_moves', 'weak_foot']

for i, col in enumerate(num_features):
    ax = axes[i // 3][i % 3]
    sns.histplot(data[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')

plt.suptitle("Figure 3.1 — Distribution Plot of Key Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix
corr_features = ['value', 'overall_rating', 'potential', 'age',
                 'height_cm', 'weight_kg', 'international_reputation',
                 'skill_moves', 'weak_foot']

corr_matrix = data[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues',
            square=True, linewidths=0.5)
plt.title("Figure 3.2 — Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print("\nTop correlations with 'value':")
print(corr_matrix['value'].sort_values(ascending=False))

## 3.5 Prepare Features and Target

In [ ]:
feature_cols = ['overall_rating', 'potential', 'age', 'height_cm', 'weight_kg',
                'international_reputation', 'skill_moves', 'weak_foot', 'position_encoded']

X = data[feature_cols]
y = data['value']

# Train-test split (80/20, stratification not applicable for regression)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardise for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{name:25s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {'Model': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'R2': round(r2, 4)}

results = []

## 3.6 Model 1: Linear Regression (Base & Polynomial)

In [ ]:
# --- Base Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
results.append(evaluate("Linear Regression", y_test, y_pred_lr))

# Actual vs Predicted Plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.4, color='steelblue', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Value (€M)")
plt.ylabel("Predicted Value (€M)")
plt.title("Figure 3.3 — Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
# --- Polynomial Regression (Degree 2) ---
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly  = poly.transform(X_test_scaled)

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly = lr_poly.predict(X_test_poly)
results.append(evaluate("Polynomial Regression", y_test, y_pred_poly))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_poly, alpha=0.4, color='darkorange', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Value (€M)")
plt.ylabel("Predicted Value (€M)")
plt.title("Figure 3.4 — Polynomial Regression (Degree 2): Actual vs Predicted")
plt.tight_layout()
plt.show()

## 3.7 Model 2: Regularisation Regression (Lasso & Ridge)

In [ ]:
# --- Lasso Regression ---
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)
results.append(evaluate("Lasso Regression", y_test, y_pred_lasso))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_lasso, alpha=0.4, color='green', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Value (€M)")
plt.ylabel("Predicted Value (€M)")
plt.title("Figure 3.5 — Lasso Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

print("\nNon-zero Lasso coefficients:")
coef_df = pd.Series(lasso.coef_, index=feature_cols)
print(coef_df[coef_df != 0].sort_values(key=abs, ascending=False))

In [ ]:
# --- Ridge Regression ---
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
results.append(evaluate("Ridge Regression", y_test, y_pred_ridge))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_ridge, alpha=0.4, color='purple', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Value (€M)")
plt.ylabel("Predicted Value (€M)")
plt.title("Figure 3.6 — Ridge Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

## 3.8 Model 3: Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(evaluate("Random Forest", y_test, y_pred_rf))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.4, color='firebrick', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Value (€M)")
plt.ylabel("Predicted Value (€M)")
plt.title("Figure 3.7 — Random Forest: Actual vs Predicted")
plt.tight_layout()
plt.show()

## 3.9 Model Evaluation Summary

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)
print("\nFigure 3.8 — Summary of Regression Models")
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'green', 'purple', 'firebrick']
models = results_df['Model']

for ax, metric, better in zip(axes, ['MAE', 'RMSE', 'R2'], ['lower', 'lower', 'higher']):
    bars = ax.bar(range(len(models)), results_df[metric], color=colors)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=30, ha='right', fontsize=8)
    ax.set_title(f'{metric} ({"↓ better" if better=="lower" else "↑ better"})')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005 * results_df[metric].max(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle("Figure 3.8 — Model Comparison (MAE, RMSE, R²)", fontsize=13)
plt.tight_layout()
plt.show()

## 3.10 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importances.plot(kind='barh', color='steelblue')
plt.xlabel("Importance")
plt.title("Figure 3.9 — Top Feature Importances (Random Forest Regressor)")
plt.tight_layout()
plt.show()

print("\nFeature Importances:")
print(importances.sort_values(ascending=False))

## 3.11 Residual Analysis (Best Model: Random Forest)

In [ ]:
residuals = y_test - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_rf, residuals, alpha=0.4, color='steelblue', s=10)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Predicted Value (€M)")
axes[0].set_ylabel("Residual (€M)")
axes[0].set_title("Figure 3.10 — Residuals vs Predicted (Random Forest)")

# Residual Distribution
sns.histplot(residuals, kde=True, ax=axes[1], color='steelblue')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel("Residual (€M)")
axes[1].set_title("Figure 3.11 — Residual Distribution (Random Forest)")

plt.tight_layout()
plt.show()